# Articulated Parts Visualizer + CD Evaluation

## Coloring rule
- **Last part index** is static parts (gray).
- **All other indices** are dynamic parts (colored).

## Sources
- **Output** (`output/MPArt90/<scene>/point_cloud/iteration_<best>/point_cloud.ply`): part labels from softmax over `weight_*`.
- **Data** (`data/<scene>/points3d.ply` + `semantics.npy`): integer label per point (since they follow a different format with no weights in PLY).

## Batch mode
- Set `RUN_ALL_SCENES = True` to write **HTML** files under `VIZ_HTML_DIR` for every scene that has the required files (no need to open 80 Plotly widgets).

In [55]:
import colorsys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import trimesh
from plyfile import PlyData
from scipy.spatial import cKDTree
import os

REPO_ROOT = Path.cwd()
OUTPUT_BASE = REPO_ROOT / "output" / "MPArt90"
DATA_BASE = REPO_ROOT / "data"

# --- batch / export ---
RUN_ALL_SCENES = False  # True: generate HTML for all scenes; False: single scene below
VIZ_HTML_DIR = OUTPUT_BASE / "_partviz_html"
VIZ_MAX_POINTS = 40_000  # subsample per cloud for speed

def collect_all_scene_names():
    scenes = [] # some don't have ply/weights, ignore these
    for d in OUTPUT_BASE.iterdir():
        if d.is_dir() and not d.name.startswith("_") and d.suffix != ".csv":
            if os.path.exists(d / "point_cloud"):
               scenes.append(d.name)
    return sorted(scenes)


all_scenes = collect_all_scene_names()  # with valid weights/pointclouds
scene_name = all_scenes[0]

print("REPO_ROOT:", REPO_ROOT)
print("scene_name:", scene_name)

REPO_ROOT: /home/stone/dev/projects/articulate/GaussianArt
scene_name: Box_102379


In [56]:
def rgb_part(k: int, num_parts: int) -> tuple[int, int, int]:
    """Last part index = gray; others = saturated distinct hues."""
    if num_parts <= 0:
        return (128, 128, 128)
    last = num_parts - 1
    if k == last:
        return (180, 180, 180)
    # Spread hues across non-static parts (golden ratio)
    hue = (k * 0.618033988749895) % 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, 0.88, 0.96)
    return (int(r * 255), int(g * 255), int(b * 255))


def plotly_scatter_parts(
    xyz: np.ndarray,
    part_id: np.ndarray,
    num_parts: int,
    title: str,
    html_path: Path | None = None,
    show: bool = True,
    rng: np.random.Generator | None = None,
    max_points: int = VIZ_MAX_POINTS,
):
    rng = rng or np.random.default_rng(0)
    n = len(xyz)
    idx = np.arange(n)
    if n > max_points:
        idx = rng.choice(n, max_points, replace=False)
    P = xyz[idx]
    pid = part_id[idx]

    fig = go.Figure()
    for k in range(num_parts):
        m = pid == k
        if not np.any(m):
            continue
        Q = P[m]
        r, g, b = rgb_part(k, num_parts)
        label = f"part {k}" + (" (last / gray)" if k == num_parts - 1 else "")
        fig.add_trace(
            go.Scatter3d(
                x=Q[:, 0],
                y=Q[:, 1],
                z=Q[:, 2],
                mode="markers",
                name=label,
                marker=dict(
                    size=2,
                    color=f"rgb({r},{g},{b})",
                    opacity=0.9,
                ),
            )
        )
    fig.update_layout(
        title=title,
        scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"),
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=50, b=0),
        height=700,
    )
    if html_path is not None:
        html_path.parent.mkdir(parents=True, exist_ok=True)
        fig.write_html(str(html_path), include_plotlyjs="cdn")
        print(f"Wrote {html_path.relative_to(REPO_ROOT)}")
    if show:
        fig.show()
    return fig


def read_best_iteration_from_results(model_dir: Path):
    p = model_dir / "results.txt"
    if not p.is_file():
        return None
    for line in p.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if s.startswith("Evaluated iteration:") or s.startswith("The best:"):
            return int(line.split(":", 1)[1].strip())
    return None


def latest_saved_iteration(model_dir: Path):
    pc = model_dir / "point_cloud"
    if not pc.is_dir():
        return None
    best = None
    for child in pc.iterdir():
        if child.is_dir() and child.name.startswith("iteration_"):
            try:
                it = int(child.name.replace("iteration_", ""))
            except ValueError:
                continue
            best = it if best is None else max(best, it)
    return best


def resolve_pred_ply_path(model_dir: Path):
    best_iter = read_best_iteration_from_results(model_dir)
    if best_iter is None:
        best_iter = latest_saved_iteration(model_dir)
    if best_iter is None:
        return None, None
    pred_ply_path = model_dir / "point_cloud" / f"iteration_{best_iter}" / "point_cloud.ply"
    if not pred_ply_path.is_file():
        iters = sorted(
            int(p.name.replace("iteration_", ""))
            for p in (model_dir / "point_cloud").glob("iteration_*")
            if p.is_dir()
        )
        if not iters:
            return None, None
        closest = min(iters, key=lambda x: abs(x - best_iter))
        best_iter = closest
        pred_ply_path = model_dir / "point_cloud" / f"iteration_{best_iter}" / "point_cloud.ply"
    return pred_ply_path, best_iter


def load_ply_vertices_and_weights(path: Path):
    plydata = PlyData.read(str(path))
    v = plydata["vertex"]
    xyz = np.vstack([v["x"], v["y"], v["z"]]).T.astype(np.float64)
    wnames = sorted(
        [n for n in v.data.dtype.names if n.startswith("weight_")],
        key=lambda s: int(s.split("_")[1]),
    )
    if not wnames:
        raise ValueError(f"No weight_* properties in {path}")
    W = np.stack([v[n] for n in wnames], axis=1).astype(np.float64)
    Ws = W - np.max(W, axis=1, keepdims=True)
    e = np.exp(Ws)
    prob = e / np.sum(e, axis=1, keepdims=True)
    part_id = np.argmax(prob, axis=1).astype(np.int32)
    num_parts = W.shape[1]
    return xyz, part_id, num_parts


def load_data_ply_and_semantics(data_dir: Path):
    """Initial point cloud + per-point integer part id from semantics.npy (no weights in PLY)."""
    ply_path = data_dir / "points3d.ply"
    sem_path = data_dir / "semantics.npy"
    if not ply_path.is_file() or not sem_path.is_file():
        return None
    plydata = PlyData.read(str(ply_path))
    v = plydata["vertex"]
    xyz = np.vstack([v["x"], v["y"], v["z"]]).T.astype(np.float64)
    sem = np.load(str(sem_path)).astype(np.int64).reshape(-1)
    if len(sem) != len(xyz):
        raise ValueError(f"semantics.npy length {len(sem)} != ply vertices {len(xyz)} for {data_dir}")
    num_parts = int(sem.max()) + 1 if sem.size else 0
    return xyz, sem, num_parts

def subsample_points(pts: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    if pts.size == 0 or len(pts) == 0:
        return pts
    if len(pts) <= n:
        return pts
    idx = rng.choice(len(pts), n, replace=False)
    return pts[idx]


def chamfer_symmetric(A: np.ndarray, B: np.ndarray) -> float:
    """Symmetric Chamfer distance between two point sets (e.g. pred vs GT)."""
    if len(A) == 0 or len(B) == 0:
        return float("nan")
    tree_B = cKDTree(B)
    tree_A = cKDTree(A)
    d_ab, _ = tree_B.query(A, k=1)  # each point in A -> nearest in B
    d_ba, _ = tree_A.query(B, k=1)  # each point in B -> nearest in A
    return 0.5 * (float(np.mean(d_ab)) + float(np.mean(d_ba)))


In [57]:
def list_output_scenes():
    return sorted(
        d.name
        for d in OUTPUT_BASE.iterdir()
        if d.is_dir() and not d.name.startswith("_") and d.suffix != ".csv"
    )


def list_data_scenes():
    return sorted(d.name for d in DATA_BASE.iterdir() if d.is_dir() and (d / "gt" / "trans.json").is_file())


rng = np.random.default_rng(0)

if RUN_ALL_SCENES:
    VIZ_HTML_DIR.mkdir(parents=True, exist_ok=True)
    # --- trained outputs ---
    for name in list_output_scenes():
        md = OUTPUT_BASE / name
        pred_path, it = resolve_pred_ply_path(md)
        if pred_path is None:
            print(f"[skip output] {name}: no point_cloud")
            continue
        try:
            xyz, pid, nparts = load_ply_vertices_and_weights(pred_path)
        except Exception as e:
            print(f"[skip output] {name}: {e}")
            continue
        plotly_scatter_parts(
            xyz,
            pid,
            nparts,
            title=f"{name} — pred iter {it} ({len(xyz)} pts, {nparts} parts)",
            html_path=VIZ_HTML_DIR / f"{name}_pred_iter{it}.html",
            show=False,
            rng=rng,
        )
    # --- data inputs ---
    for name in list_data_scenes():
        dd = DATA_BASE / name
        loaded = load_data_ply_and_semantics(dd)
        if loaded is None:
            print(f"[skip input] {name}: missing points3d.ply or semantics.npy")
            continue
        xyz, sem, nparts = loaded
        plotly_scatter_parts(
            xyz,
            sem,
            nparts,
            title=f"{name} — data input ({len(xyz)} pts, semantics 0..{nparts-1})",
            html_path=VIZ_HTML_DIR / f"{name}_input.html",
            show=False,
            rng=rng,
        )
    print(f"Done. Open HTML files in: {VIZ_HTML_DIR}")
else:
    model_dir = OUTPUT_BASE / scene_name
    data_dir = DATA_BASE / scene_name
    pred_path, best_iter = resolve_pred_ply_path(model_dir)
    if pred_path is None:
        raise FileNotFoundError(f"No prediction PLY for {scene_name}")
    xyz, pid, nparts = load_ply_vertices_and_weights(pred_path)
    print(f"Pred: {pred_path.relative_to(REPO_ROOT)}  parts={nparts}")
    plotly_scatter_parts(
        xyz,
        pid,
        nparts,
        title=f"{scene_name} — pred iter {best_iter}",
        html_path=None,
        show=True,
        rng=rng,
    )

    loaded = load_data_ply_and_semantics(data_dir)
    if loaded is None:
        print("No data input (points3d.ply + semantics.npy) for this scene.")
    else:
        dxyz, sem, snparts = loaded
        plotly_scatter_parts(
            dxyz,
            sem,
            snparts,
            title=f"{scene_name} — data input (semantics.npy)",
            html_path=None,
            show=True,
            rng=rng,
        )

Pred: output/MPArt90/Box_102379/point_cloud/iteration_78000/point_cloud.ply  parts=2


## Optional: per-part Chamfer (single scene)

Run when `RUN_ALL_SCENES` is False. **GT** is `data/<scene>/points3d.ply` with **`semantics.npy`** (same part indices as training). **Pred** is the Gaussian `point_cloud.ply` from softmax over `weight_*`. Per-part Chamfer is symmetric NN distance between those point sets.


In [58]:
GT_SAMPLES_PER_PART = 20000
PRED_CAP = 20000

if not RUN_ALL_SCENES:
    gt_loaded = load_data_ply_and_semantics(data_dir)
    if gt_loaded is None:
        print("Chamfer: need data/<scene>/points3d.ply + semantics.npy for GT.")
    else:
        gt_xyz, gt_sem, snparts = gt_loaded
        n_part = max(nparts, snparts)
        last = n_part - 1
        cds = {}
        for k in range(n_part):
            P = subsample_points(xyz[pid == k], PRED_CAP, rng)
            G = subsample_points(gt_xyz[gt_sem == k], GT_SAMPLES_PER_PART, rng)
            cds[k] = chamfer_symmetric(P, G)
            role = "static" if k == last else "dynamic"
            print(f"Part {k} ({role}) Chamfer pred↔GT semantics: {cds[k]:.6f}")
        dyn_vals = [cds[k] for k in range(last) if not np.isnan(cds[k])]
        if dyn_vals:
            print(f"Mean Chamfer (dynamic parts 0..{last - 1}): {float(np.mean(dyn_vals)):.6f}")
        if last in cds and not np.isnan(cds[last]):
            print(f"Static part ({last}) Chamfer: {cds[last]:.6f}")
else:
    print("Chamfer cell skipped in RUN_ALL_SCENES mode.")


Part 0 Chamfer: 0.000000
Part 1 Chamfer: 0.000000
Mean: 0.0


## Optional: export part Chamfer figures + summary JSON

Set `EXPORT_PART_CD_REPORT = True`. Writes under **`output/MPArt90/<scene>/part_chamfer/`**:
- `figures/part_XX.png` — GT (left) vs pred (right) for each part, with CD in the title (last part labeled static).
- `cd_summary.json` — per-part CD, mean over **dynamic** parts (all indices except the last), and static part CD.

Set `EXPORT_PART_CD_ALL_SCENES = True` to loop `collect_all_scene_names()`; otherwise only `scene_name`.


In [ ]:
import json as json_module

# --- set True to write PNGs + cd_summary.json per scene ---
EXPORT_PART_CD_REPORT = False
EXPORT_PART_CD_ALL_SCENES = False
PART_CD_MAX_POINTS_VIZ = 8000
PART_CD_PRED_CAP = 20000
PART_CD_GT_CAP = 20000


def _scatter3d(ax, pts: np.ndarray, color, title: str):
    if len(pts):
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=1, c=color, alpha=0.65, depthshade=False)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")


def export_part_chamfer_report(
    scene: str,
    model_dir: Path,
    data_dir: Path,
    rng: np.random.Generator,
):
    pred_path, it = resolve_pred_ply_path(model_dir)
    if pred_path is None:
        return None, "no prediction point_cloud.ply"
    loaded = load_data_ply_and_semantics(data_dir)
    if loaded is None:
        return None, "missing points3d.ply or semantics.npy"
    pxyz, pid, nparts_p = load_ply_vertices_and_weights(pred_path)
    gxyz, gsem, nparts_g = loaded
    n_part = max(nparts_p, nparts_g)
    last = n_part - 1

    out_dir = model_dir / "part_chamfer"
    fig_dir = out_dir / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    cds = {}
    for k in range(n_part):
        P = subsample_points(pxyz[pid == k], PART_CD_PRED_CAP, rng)
        G = subsample_points(gxyz[gsem == k], PART_CD_GT_CAP, rng)
        cds[k] = chamfer_symmetric(P, G)

        Pv = subsample_points(pxyz[pid == k], PART_CD_MAX_POINTS_VIZ, rng)
        Gv = subsample_points(gxyz[gsem == k], PART_CD_MAX_POINTS_VIZ, rng)
        fig = plt.figure(figsize=(12, 5))
        ax1 = fig.add_subplot(121, projection="3d")
        ax2 = fig.add_subplot(122, projection="3d")
        _scatter3d(ax1, Gv, "#888888", "GT (semantics.npy)")
        r, g, b = rgb_part(k, n_part)
        _scatter3d(ax2, Pv, (r / 255.0, g / 255.0, b / 255.0), "Pred (softmax weights)")
        cd = cds[k]
        lab = "static" if k == last else "dynamic"
        fig.suptitle(f"{scene}  part {k} ({lab})  Chamfer pred↔GT = {cd:.6f}")
        allv = []
        if len(Gv):
            allv.append(Gv)
        if len(Pv):
            allv.append(Pv)
        if allv:
            U = np.vstack(allv)
            cmin, cmax = U.min(0), U.max(0)
            span = np.maximum(cmax - cmin, 1e-9)
            pad = span * 0.05
            for ax in (ax1, ax2):
                ax.set_xlim(cmin[0] - pad[0], cmax[0] + pad[0])
                ax.set_ylim(cmin[1] - pad[1], cmax[1] + pad[1])
                ax.set_zlim(cmin[2] - pad[2], cmax[2] + pad[2])
        fig.tight_layout()
        fig.savefig(fig_dir / f"part_{k:02d}.png", dpi=150)
        plt.close(fig)

    dyn_vals = [cds[k] for k in range(last) if not np.isnan(cds[k])]
    mean_dyn = float(np.mean(dyn_vals)) if dyn_vals else float("nan")
    summary = {
        "scene": scene,
        "pred_iteration": it,
        "num_parts": n_part,
        "static_part_index": last,
        "chamfer_per_part": {str(k): float(cds[k]) if not np.isnan(cds[k]) else None for k in sorted(cds)},
        "mean_chamfer_dynamic_parts": mean_dyn,
        "chamfer_static_part": float(cds[last]) if last in cds and not np.isnan(cds[last]) else None,
    }
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "cd_summary.json").write_text(json_module.dumps(summary, indent=2), encoding="utf-8")
    return summary, None


if EXPORT_PART_CD_REPORT:
    rng_cd = np.random.default_rng(1)
    scenes = collect_all_scene_names() if EXPORT_PART_CD_ALL_SCENES else [scene_name]
    for sn in scenes:
        md = OUTPUT_BASE / sn
        dd = DATA_BASE / sn
        summ, err = export_part_chamfer_report(sn, md, dd, rng_cd)
        if err:
            print(f"[skip] {sn}: {err}")
        else:
            print(f"{sn}: wrote {md / 'part_chamfer'}")
else:
    pass  # set EXPORT_PART_CD_REPORT = True to generate per-scene part_chamfer/


In [65]:
# --- Joint axes: save PNGs (GT + pred), tab10 color per joint index ---
# Fixes: `from argparse import Namespace` before eval(cfg_args). No Open3D windows.

from argparse import Namespace
import glob
import re
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

REPO = Path("/home/stone/dev/projects/articulate/GaussianArt").resolve()
sys.path.insert(0, str(REPO))

from eval_axis import interpret_transforms, read_gt
from utils.general_utils import build_rotation

MODEL_PATH = REPO / "output/MPArt90/Table_31249"
ITERATION = None
OUT_DIR = MODEL_PATH / "axis_viz"


def _best_iter(model_path: Path):
    r = model_path / "results.txt"
    if not r.is_file():
        return None
    line = r.read_text().splitlines()[0]
    m = re.match(r"(?:Evaluated iteration|The best):\s*(\d+)", line.strip())
    return int(m.group(1)) if m else None


def _joint_type_heuristic(cur_R: np.ndarray) -> str:
    return "prismatic" if np.abs(cur_R - np.eye(3)).mean() < 5e-2 else "revolute"


def _tab10(i: int):
    c = plt.cm.tab10(i % 10)
    return (float(c[0]), float(c[1]), float(c[2]))


def _plot_axes(origins, directions, axis_len, title, out_path: Path):
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")
    all_p = []
    for i, (o, d) in enumerate(zip(origins, directions)):
        o = np.asarray(o, dtype=np.float64).reshape(3)
        d = np.asarray(d, dtype=np.float64).reshape(3)
        n = np.linalg.norm(d)
        if n < 1e-9:
            continue
        d = d / n
        col = _tab10(i)
        p1 = o + axis_len * d
        ax.plot([o[0], p1[0]], [o[1], p1[1]], [o[2], p1[2]], color=col, linewidth=2.5, label=f"joint {i}")
        ax.scatter([o[0]], [o[1]], [o[2]], color=[col], s=45)
        all_p.extend([o, p1])
    L = axis_len * 0.35
    ax.quiver(0, 0, 0, L, 0, 0, color="r", arrow_length_ratio=0.12)
    ax.quiver(0, 0, 0, 0, L, 0, color="g", arrow_length_ratio=0.12)
    ax.quiver(0, 0, 0, 0, 0, L, color="b", arrow_length_ratio=0.12)
    if all_p:
        U = np.stack(all_p, axis=0)
        cmin, cmax = U.min(0), U.max(0)
        span = np.maximum(cmax - cmin, 1e-9)
        pad = span * 0.15
        ax.set_xlim(cmin[0] - pad[0], cmax[0] + pad[0])
        ax.set_ylim(cmin[1] - pad[1], cmax[1] + pad[1])
        ax.set_zlim(cmin[2] - pad[2], cmax[2] + pad[2])
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


cfg = MODEL_PATH / "cfg_args"
ns = eval(cfg.read_text())
source_path = Path(ns.source_path).resolve()

gt_joints = read_gt(str(source_path / "gt" / "trans.json"))
gt_o = [np.asarray(g["axis_position"], dtype=np.float64).reshape(3) for g in gt_joints]
gt_d = [np.asarray(g["axis_direction"], dtype=np.float64).reshape(3) for g in gt_joints]

it = ITERATION if ITERATION is not None else _best_iter(MODEL_PATH)
if it is None:
    cks = sorted(glob.glob(str(MODEL_PATH / "ckpts" / "ours_*.pth")))
    it = int(Path(cks[-1]).stem.split("_")[-1])

ckpt = torch.load(MODEL_PATH / "ckpts" / f"ours_{it}.pth", map_location="cpu")
art_R = ckpt["articulation_params"]["art_R"].cuda()
art_T = ckpt["articulation_params"]["art_T"].cuda()
articulation_matrix = build_rotation(art_R).detach().cpu().numpy()
art_T_np = art_T.detach().cpu().numpy()
N = art_R.shape[0]

pr_o, pr_d = [], []
for i in range(0, N - 1):
    cur_R, cur_T = articulation_matrix[i], art_T_np[i]
    jt = _joint_type_heuristic(cur_R)
    pred_joint, _, _ = interpret_transforms(np.eye(3), np.zeros(3), cur_R, cur_T, joint_type=jt)
    pr_o.append(np.asarray(pred_joint["axis_position"], dtype=np.float64).reshape(3))
    pr_d.append(np.asarray(pred_joint["axis_direction"], dtype=np.float64).reshape(3))

all_pts = np.stack(gt_o + pr_o, axis=0)
scale = float(max(0.15, np.ptp(all_pts, axis=0).max() * 0.25))

_plot_axes(gt_o, gt_d, scale, f"GT axes — {source_path.name}", OUT_DIR / "gt_axes.png")
_plot_axes(pr_o, pr_d, scale, f"Pred axes — {MODEL_PATH.name} (iter {it})", OUT_DIR / "pred_axes.png")
print(f"Wrote {OUT_DIR / 'gt_axes.png'}")
print(f"Wrote {OUT_DIR / 'pred_axes.png'}")


Wrote /home/stone/dev/projects/articulate/GaussianArt/output/MPArt90/Table_31249/axis_viz/gt_axes.png
Wrote /home/stone/dev/projects/articulate/GaussianArt/output/MPArt90/Table_31249/axis_viz/pred_axes.png


In [66]:
# --- Match pred joint indices to GT by minimum axis-direction angle (Hungarian) ---
# Run the previous cell first (`articulation_matrix`, `art_T_np`, `gt_joints`, `interpret_transforms`).

import numpy as np
from scipy.optimize import linear_sum_assignment


def _joint_type_heuristic(cur_R: np.ndarray) -> str:
    return "prismatic" if np.abs(cur_R - np.eye(3)).mean() < 5e-2 else "revolute"


def _unit(v):
    v = np.asarray(v, dtype=np.float64).reshape(3)
    n = np.linalg.norm(v)
    if n < 1e-12:
        return v, 0.0
    return v / n, n


def _axis_angle_deg(a_d, b_d):
    """Smallest angle between two lines (|cos| treats opposite direction as same)."""
    a_d, na = _unit(a_d)
    b_d, nb = _unit(b_d)
    if na < 1e-12 or nb < 1e-12:
        return np.nan
    c = np.clip(np.abs(float(np.dot(a_d, b_d))), 0.0, 1.0)
    return float(np.rad2deg(np.arccos(c)))


def test_pred_gt_axis_indices(articulation_matrix, art_T_np, gt_joints):
    """Print cost matrix [deg] and Hungarian assignment pred_i -> gt_j."""
    N = articulation_matrix.shape[0]
    num_movable = N - 1
    num_gt = len(gt_joints)
    preds = []
    for i in range(num_movable):
        cur_R = articulation_matrix[i]
        jt = _joint_type_heuristic(cur_R)
        res, _, _ = interpret_transforms(
            np.eye(3), np.zeros(3), cur_R, art_T_np[i], joint_type=jt
        )
        preds.append(res)

    cost_matrix = np.full((num_movable, num_gt), np.nan, dtype=np.float64)
    for i in range(num_movable):
        for j in range(num_gt):
            cost_matrix[i, j] = _axis_angle_deg(
                preds[i]["axis_direction"], gt_joints[j]["axis_direction"]
            )

    if np.any(np.isnan(cost_matrix)):
        bad = np.argwhere(np.isnan(cost_matrix))
        print("Degenerate axis_direction (NaN cost) at pred,gt pairs:", bad.tolist())

    large = 180.0
    cost_for_solver = np.where(np.isfinite(cost_matrix), cost_matrix, large)

    row_ind, col_ind = linear_sum_assignment(cost_for_solver)

    np.set_printoptions(precision=2, suppress=True, linewidth=120)
    print("Cost matrix [deg]: rows=pred 0..{}, cols=GT 0..{}".format(num_movable - 1, num_gt - 1))
    print(cost_matrix)
    print()
    print("Hungarian assignment (pred_idx -> gt_idx), angle [deg]:")
    for r, c in zip(row_ind, col_ind):
        ang = cost_matrix[r, c]
        jt = _joint_type_heuristic(articulation_matrix[r])
        print(f"  pred {r:2d} -> gt {c:2d}   angle={ang:7.3f}   joint_type={jt}")
    print()
    print("pred_indices =", row_ind.tolist(), "  gt_indices =", col_ind.tolist())


test_pred_gt_axis_indices(articulation_matrix, art_T_np, gt_joints)


Cost matrix [deg]: rows=pred 0..3, cols=GT 0..3
[[ 0.07  0.07 89.99 89.99]
 [ 0.05  0.05 90.   90.  ]
 [20.7  20.7  84.32 84.32]
 [46.61 46.61 71.92 71.92]]

Hungarian assignment (pred_idx -> gt_idx), angle [deg]:
  pred  0 -> gt  0   angle=  0.065   joint_type=prismatic
  pred  1 -> gt  1   angle=  0.055   joint_type=prismatic
  pred  2 -> gt  2   angle= 84.321   joint_type=prismatic
  pred  3 -> gt  3   angle= 71.916   joint_type=revolute

pred_indices = [0, 1, 2, 3]   gt_indices = [0, 1, 2, 3]
